# LangGraph Basics — AI Engineering Interview Prep

StateGraph, nodes, conditional edges, and agent loops.

**Install**: `pip install langgraph langchain-anthropic`

In [ ]:
import os
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    anthropic_api_key=os.environ.get("ANTHROPIC_API_KEY"),
    max_tokens=400
)

print("LangGraph ready")

## 1. Basic StateGraph — Linear Flow

In [ ]:
# State definition
class ReviewState(TypedDict):
    code: str
    issues: str
    fix: str
    final_review: str

# Node functions
def detect_issues(state: ReviewState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a code reviewer. List only critical issues. Max 3 bullet points."),
        HumanMessage(content=f"Review this code:\n{state['code']}")
    ])
    return {"issues": response.content}

def generate_fix(state: ReviewState) -> dict:
    response = llm.invoke([
        HumanMessage(content=f"Fix these issues in the code:\nCode: {state['code']}\nIssues: {state['issues']}\nReturn fixed code only.")
    ])
    return {"fix": response.content}

def final_review(state: ReviewState) -> dict:
    response = llm.invoke([
        HumanMessage(content=f"Confirm the fix addresses all issues. One sentence verdict.\nIssues: {state['issues']}\nFix: {state['fix']}")
    ])
    return {"final_review": response.content}

# Build graph
graph = StateGraph(ReviewState)
graph.add_node("detect_issues", detect_issues)
graph.add_node("generate_fix", generate_fix)
graph.add_node("final_review", final_review)

graph.set_entry_point("detect_issues")
graph.add_edge("detect_issues", "generate_fix")
graph.add_edge("generate_fix", "final_review")
graph.add_edge("final_review", END)

app = graph.compile()

# Run
buggy_code = """
def compute_accuracy(predictions, labels):
    correct = 0
    for i in range(len(predictions)):
        if predictions[i] = labels[i]:  # bug: assignment not comparison
            correct += 1
    return correct / len(predictions)  # bug: no zero-division check
"""

result = app.invoke({"code": buggy_code, "issues": "", "fix": "", "final_review": ""})
print("Issues detected:")
print(result["issues"])
print("\nFinal review:")
print(result["final_review"])

## 2. Conditional Edges — Branch Logic

In [ ]:
class ClassifierState(TypedDict):
    question: str
    category: str
    answer: str

def classify_question(state: ClassifierState) -> dict:
    response = llm.invoke([
        HumanMessage(content=f"Classify this ML question as exactly one of: code, theory, math\nQuestion: {state['question']}\nOutput ONLY the category word.")
    ])
    return {"category": response.content.strip().lower()}

def answer_code(state: ClassifierState) -> dict:
    response = llm.invoke([HumanMessage(content=f"Answer with a Python code example (max 8 lines):\n{state['question']}")] )
    return {"answer": response.content}

def answer_theory(state: ClassifierState) -> dict:
    response = llm.invoke([HumanMessage(content=f"Answer with a clear conceptual explanation (max 60 words):\n{state['question']}")] )
    return {"answer": response.content}

def answer_math(state: ClassifierState) -> dict:
    response = llm.invoke([HumanMessage(content=f"Answer with mathematical notation and formulas:\n{state['question']}")] )
    return {"answer": response.content}

def route_question(state: ClassifierState) -> Literal["answer_code", "answer_theory", "answer_math"]:
    cat = state["category"]
    if "code" in cat: return "answer_code"
    elif "math" in cat: return "answer_math"
    return "answer_theory"

# Build conditional graph
g = StateGraph(ClassifierState)
g.add_node("classify", classify_question)
g.add_node("answer_code", answer_code)
g.add_node("answer_theory", answer_theory)
g.add_node("answer_math", answer_math)

g.set_entry_point("classify")
g.add_conditional_edges("classify", route_question)
for node in ["answer_code", "answer_theory", "answer_math"]:
    g.add_edge(node, END)

router_app = g.compile()

questions = [
    "How do I implement k-means in Python?",
    "What is the intuition behind attention mechanisms?",
    "Derive the gradient of cross-entropy loss with respect to softmax output.",
]

for q in questions:
    result = router_app.invoke({"question": q, "category": "", "answer": ""})
    print(f"Q: {q}")
    print(f"Category: {result['category']}")
    print(f"Answer: {result['answer'].strip()[:200]}")
    print("-" * 60)

## 3. Conversation Loop with Message History

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot_node(state: ChatState) -> dict:
    system = SystemMessage(content="You are a concise ML interview coach. Max 60 words per answer.")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}

chat_graph = StateGraph(ChatState)
chat_graph.add_node("chatbot", chatbot_node)
chat_graph.set_entry_point("chatbot")
chat_graph.add_edge("chatbot", END)
chat_app = chat_graph.compile()

# Simulate multi-turn conversation
conversation_history = []
turns = [
    "What is regularization?",
    "What are the two most common types?",
    "Which one promotes sparsity and why?",
]

for user_msg in turns:
    conversation_history.append(HumanMessage(content=user_msg))
    state = chat_app.invoke({"messages": conversation_history})
    ai_response = state["messages"][-1]
    conversation_history.append(ai_response)
    print(f"User: {user_msg}")
    print(f"Bot: {ai_response.content.strip()}")
    print()

## Interview Tips

- **StateGraph vs Chain**: LangGraph handles branching, loops, and state; LangChain LCEL is for linear pipelines.
- **State**: a TypedDict that persists across nodes. Each node reads state and returns a partial update.
- **Conditional edges**: router function returns the next node name — enables if/else branching.
- **`add_messages` reducer**: merges message lists across nodes rather than overwriting. Essential for chat.
- **Cycles**: LangGraph supports loops (node → node → same node). Use for retry/reflection patterns.
- **Checkpointing**: save/restore state with `checkpointer=MemorySaver()` for long-running agents.